In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. OpenPaved ###

In [4]:
iters = np.shape(P_atm)[0] # total timestep.

In [5]:
CP_measure = 0 # we do not consider 'measure' for the time being.

In [6]:
class OpenPaved:
    def __init__(self, init_instor, intstorcap_openpaved = 1.6, stormfrac_openpaved = 1.0, discfrac_openpaved = 0.0, infilcap_openpaved = 1):
        
        # state
        self.init_instor = init_instor
        
        # parameter
        self.intstorcap = intstorcap_openpaved
        self.stormfrac = stormfrac_openpaved
        self.discfrac = discfrac_openpaved
        self.infilcap = infilcap_openpaved

    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are input information.'
    
    def mxd_frac(self):
        return 1 - self.stormfrac
    
    def sol(self, p_atm, e_pot_ow):
        intcp = min(self.intstorcap, max(0, p_atm + self.init_instor))
        e_atm = min(e_pot_ow, intcp)
        intstor = intcp - e_atm
        p_gw = max(0, min(p_atm - (self.intstorcap - self.init_instor), self.infilcap * delta_t)) # infiltration capacity (mm/d) * time step size (hr to d)
        r_swds = self.stormfrac * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor) - p_gw)
        r_mss = self.mxd_frac() * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor) - p_gw)
        r_up = self.discfrac * max(0, p_atm - e_atm - (intstor - self.init_instor) - p_gw)
        
        # update state
        self.init_instor = intstor
        
        return intcp, e_atm, intstor, p_gw, r_swds, r_mss, r_up

In [7]:
delta_t = 1 / 24

t = 1
E_atm = [0]
Intcp = [0]
IntStor = [0]
P_gw = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

# Give initial interception storage.
init_instor_t0 = 0 

# Specify the parameter or use the default setting.
m = OpenPaved(init_instor_t0, intstorcap_openpaved = 1.6, stormfrac_openpaved = 1.0, discfrac_openpaved = 0.0, infilcap_openpaved = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    P_gw.append(sol[3])
    R_swds.append(sol[4])
    R_mss.append(sol[5])
    R_up.append(sol[6])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_OpenPaved.csv'
np.savetxt('sol/' + filename, np.c_[Intcp, E_atm, IntStor, P_gw, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, P_gw, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


### Conclusion.


__Note that__:
In the excel file, there is a small mistake:
In the measure area input data form:
If total measure area is 0, then inflow area to measure is 0.

### For data preparation of unpaved unit buildup

1. C2S1: discfrac = 0.55, stormfrac = 0.75, get R_up:

In [8]:
delta_t = 1 / 24

t = 1
E_atm = [0]
Intcp = [0]
IntStor = [0]
P_gw = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

# Give initial interception storage.
init_instor_t0 = 0 

# Specify the parameter or use the default setting.
m = OpenPaved(init_instor_t0, intstorcap_openpaved = 1.6, stormfrac_openpaved = 0.75, discfrac_openpaved = 0.55, infilcap_openpaved = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    P_gw.append(sol[3])
    R_swds.append(sol[4])
    R_mss.append(sol[5])
    R_up.append(sol[6])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_OpenPaved_forunpavedbuildup_c2s1_disfrac055stormfrac075.csv'
np.savetxt('sol/' + filename, np.c_[Intcp, E_atm, IntStor, P_gw, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, P_gw, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.
